In [ ]:
# ===============================
# Colab script to read the ChG-InterDecagon targets file
# ===============================

import pandas as pd
from io import BytesIO
from google.colab import files

# OPTION 1: upload the file from your computer
# Uncomment this block if you want to upload manually

print("If you have the file locally (e.g. ChG-InterDecagon_targets.csv.gz), upload it now.")
uploaded = files.upload()  # this opens a dialog in Colab


If you have the file locally (e.g. ChG-InterDecagon_targets.csv.gz), upload it now.


Saving ChG-InterDecagon_targets.csv.gz to ChG-InterDecagon_targets.csv.gz


In [ ]:
# Get the first (and usually only) uploaded file name
file_name = list(uploaded.keys())[0]
print(f"File uploaded: {file_name}")

# The file from SNAP is a compressed CSV (gzip) where each row looks like: CID000060752,3757
# but the first row may contain a header like "# Drug\tGene" that we want to ignore.
# We read it as CSV with a comma separator and build correct column names.

# Read the file
df = pd.read_csv(file_name, sep=",", header=None, names=["Drug", "Gene"])

# Sometimes the first row is not a real data row but a header artifact.
# We can detect it and drop it if needed.
if df.iloc[0]["Drug"].startswith("#"):
    print("Detected header artifact in the first row. Dropping it.")
    df = df.iloc[1:].reset_index(drop=True)

# Convert Gene to numeric (it may be read as float because of the artifact row)
df["Gene"] = pd.to_numeric(df["Gene"], errors="coerce")

print("Preview of the dataset:")
print(df.head())

print("\nDataset shape (rows, columns):", df.shape)
print(f"Number of unique drugs: {df['Drug'].nunique()}")
print(f"Number of unique genes: {df['Gene'].nunique()}")

# OPTIONAL: show basic description
print("\nBasic info:")
print(df.info())

File uploaded: ChG-InterDecagon_targets.csv.gz
Detected header artifact in the first row. Dropping it.
Preview of the dataset:
           Drug    Gene
0  CID000060752  3757.0
1  CID006918155  2908.0
2  CID103052762  3359.0
3  CID023668479  1230.0
4  CID000028864  1269.0

Dataset shape (rows, columns): (131034, 2)
Number of unique drugs: 1774
Number of unique genes: 7795

Basic info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131034 entries, 0 to 131033
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Drug    131034 non-null  object 
 1   Gene    131034 non-null  float64
dtypes: float64(1), object(1)
memory usage: 2.0+ MB
None


# Preprocessing
- Remove corrupted or header rows
- Convert data types
- Remove duplicates
- Normalize identifiers

In [ ]:
# Remove header-like artifacts or invalid entries
df = df[~df["Drug"].str.startswith("#")]
df = df.dropna(subset=["Drug", "Gene"])

# Ensure columns have the correct types:
df["Drug"] = df["Drug"].astype(str).str.strip()
df["Gene"] = pd.to_numeric(df["Gene"], errors="coerce").astype(float)

# Remove duplicates
df = df.drop_duplicates(subset=["Drug", "Gene"])

# Normalize identifiers
# Drug codes (CIDs...) may contain leading zeros or inconsistent formats
df["Drug"] = df["Drug"].str.replace("CID", "").astype(int)

# Fianal check
print(df.info())
print(df.head())
print(f"Total unique drugs: {df['Drug'].nunique()}")
print(f"Total unique genes: {df['Gene'].nunique()}")
print(f"Total interactions: {len(df)}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131034 entries, 0 to 131033
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Drug    131034 non-null  int64  
 1   Gene    131034 non-null  float64
dtypes: float64(1), int64(1)
memory usage: 2.0 MB
None
        Drug    Gene
0      60752  3757.0
1    6918155  2908.0
2  103052762  3359.0
3   23668479  1230.0
4      28864  1269.0
Total unique drugs: 1774
Total unique genes: 7795
Total interactions: 131034


# Drug-Gene Network
